In [1]:
#IMPORTANT ***RUN THIS FIRST***
!pip install pandas plotly dash networkx
!pip install dash==3.2.0 plotly

   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ------- -------------------------------- 1.6/8.4 MB 7.7 MB/s eta 0:00:01
   ------------------ --------------------- 3.9/8.4 MB 9.6 MB/s eta 0:00:01
   ----------------------------- ---------- 6.3/8.4 MB 10.0 MB/s eta 0:00:01
   ---------------------------------------  8.4/8.4 MB 10.2 MB/s eta 0:00:01
   ---------------------------------------- 8.4/8.4 MB 9.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 9.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/6.9 MB ? eta -:--:--
   ------------- -------------------------- 2.4/6.9 MB 11.4 MB/s eta 0:00:01
   --------------------------- ------------ 4.7/6.9 MB 11.2 MB/s eta 0:00:01
   ---------------------------------------  6.8/6.9 MB 11.0 MB/s eta 0:00:01
   ---------------------------------------- 6.9/6.9 MB 10.2 MB/s eta 0:00:00

  Attempting uninstall: ty

In [2]:
# NECESSARY LIBARIES + FRAMEWORKS

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

import dash
from dash import dcc, html, Input, Output, Dash
import pandas as pd
import plotly.express as px

# Load dataset
df = pd.read_csv("Provisional_COVID-19_Deaths_by_Place_of_Death_and_State_20260417.csv")
df.columns = df.columns.str.strip()


In [3]:
#Jeanne

#BUBBLE CHART (INTERACTIVE VISUALIZATION #1)

df = pd.read_csv("Provisional_COVID-19_Deaths_by_Place_of_Death_and_State_20260417.csv")
df.columns = df.columns.str.strip()

# Year column
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df = df[df["Year"].between(2020, 2023)]

# From states to regions
region_map = {
    # Northeast
    "New York":"Northeast","New Jersey":"Northeast","Pennsylvania":"Northeast",
    "Massachusetts":"Northeast","Connecticut":"Northeast","Rhode Island":"Northeast",
    "Vermont":"Northeast","New Hampshire":"Northeast","Maine":"Northeast",

    # Midwest
    "Illinois":"Midwest","Ohio":"Midwest","Michigan":"Midwest","Indiana":"Midwest",
    "Wisconsin":"Midwest","Minnesota":"Midwest","Iowa":"Midwest","Missouri":"Midwest",
    "Kansas":"Midwest","Nebraska":"Midwest","South Dakota":"Midwest","North Dakota":"Midwest",

    # South
    "Texas":"South","Florida":"South","Georgia":"South","North Carolina":"South",
    "Virginia":"South","Tennessee":"South","Alabama":"South","South Carolina":"South",
    "Kentucky":"South","Louisiana":"South","Mississippi":"South","Arkansas":"South",
    "Oklahoma":"South","West Virginia":"South","Maryland":"South","Delaware":"South",
    "District of Columbia":"South",

    # West
    "California":"West","Washington":"West","Oregon":"West","Nevada":"West",
    "Arizona":"West","Utah":"West","Colorado":"West","New Mexico":"West",
    "Idaho":"West","Montana":"West","Wyoming":"West","Alaska":"West","Hawaii":"West"
}

df["region"] = df["State"].map(region_map)
df = df.dropna(subset=["region"])

# Keep only needed columns
df = df[[
    "region", "Year",
    "COVID-19 Deaths",
    "Pneumonia Deaths",
    "Influenza Deaths"
]]

# Convert to numeric values
for col in ["COVID-19 Deaths", "Pneumonia Deaths", "Influenza Deaths"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna()

# Reshape (long format)
df_long = df.melt(
    id_vars=["region", "Year"],
    value_vars=["COVID-19 Deaths", "Pneumonia Deaths", "Influenza Deaths"],
    var_name="disease",
    value_name="deaths"
)

# Clean disease names
df_long["disease"] = df_long["disease"].str.replace(" Deaths", "", regex=False)
df_long = df_long.groupby(["region", "Year", "disease"])["deaths"].sum().reset_index() #aggregation

regions = ["Northeast", "Midwest", "South", "West"] #regions for slider

# Dash App
app = Dash(__name__)
app.layout = html.Div([
    html.H3(("US Disease Deaths Bubble Chart (2020–2023)"),
        style = {"color": "white"}
    ),

    dcc.Slider(
        id="region-slider",
        min=0,
        max=len(regions)-1,
        step=1,
        value=0,
        marks={
            i: {"label": regions[i], "style": {"color": "white", "font-weight": "bold"}}
           for i in range(len(regions))
        }
    ),

    dcc.Graph(id="bubble-chart")
])

# Callback
@app.callback(
    Output("bubble-chart", "figure"),
    Input("region-slider", "value")
)
def update_chart(region_index):

    selected_region = regions[region_index]
    filtered = df_long[df_long["region"] == selected_region]

    fig = px.scatter(
        filtered,
        x="deaths",
        y="Year",
        size="deaths",
        color="disease",
        size_max=70,
        title=f"{selected_region}: COVID-19, Pneumonia, Influenza Deaths",
        category_orders={"disease": ["COVID-19", "Pneumonia", "Influenza"]}
    )

    fig.update_layout(
        xaxis=dict(title="Number of Deaths", range=[0, 100000]),
        yaxis=dict(title="Year", range=[2019.5, 2023.5]),
        template="plotly_white"
    )

    return fig

if __name__ == "__main__":
    app.run(jupyter_mode="inline")

In [4]:
  #BAR CHART (INTERACTIVE VISUALIZATION #2)

import dash
from dash import dcc, html, Input, Output, Dash
import pandas as pd
import plotly.express as px

df = pd.read_csv("Provisional_COVID-19_Deaths_by_Place_of_Death_and_State_20260417.csv")
df.columns = df.columns.str.strip()

df["COVID-19 Deaths"] = pd.to_numeric(df["COVID-19 Deaths"], errors="coerce")   # Convert numeric

df = df[df["Year"].between(2020, 2023)]   # Filter years 2020–2023
df = df[["State", "Year", "COVID-19 Deaths"]].dropna() # Relevant columns

# From states to regions
df["Region"] = df["State"].map(region_map)
df = df.dropna(subset=["Region"])

# Aggregate total covid deaths by state
df_agg = df.groupby(["Region", "State"])["COVID-19 Deaths"].sum().reset_index()

# Dash App
app = Dash(__name__)

app.layout = html.Div([
    html.H3(("COVID-19 Deaths by State (Region-Based)"),
       style = {"color": "white"}
    ),

    html.Label(("Select Region:"),
    style={"color": "white"}

    ),
    dcc.Dropdown(
        id="region-dropdown",
        options=[
            {"label": "Northeast", "value": "Northeast"},
            {"label": "Midwest", "value": "Midwest"},
            {"label": "South", "value": "South"},
            {"label": "West", "value": "West"}
        ],
        value="Northeast",
        clearable=False,
        style = {"backgroundColor": "#e6f1f9"}
    ),

    dcc.Graph(id="bar-chart")
])

# Callback
@app.callback(
    Output("bar-chart", "figure"),
    Input("region-dropdown", "value")
)
def update_bar(selected_region):

    dff = df_agg[df_agg["Region"] == selected_region]

    # Sort (descending order)
    dff = dff.sort_values(by="COVID-19 Deaths", ascending=True)

    fig = px.bar(
        dff,
        x="COVID-19 Deaths",
        y="State",
        orientation="h",
        title=f"COVID-19 Deaths by State — {selected_region}",
        text="COVID-19 Deaths"
    )

    fig.update_layout(
        xaxis=dict(title="Number of Deaths", range=[0, 100000]),
        yaxis=dict(title="States"),
        template="plotly_white",
        height=700
    )

    fig.update_traces(textposition="outside")

    return fig

if __name__ == "__main__":
    app.run(jupyter_mode="inline", port = 9006)